1 — Setup کامل

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv

PROJECT_ROOT = Path(".")

GRAPH_DIR = PROJECT_ROOT / "Data_ml" / "graph_dataset"
DAY13_DIR = GRAPH_DIR / "day13_explainability"
MODEL_DIR = DAY13_DIR / "saved_models"

DAY16_DIR = GRAPH_DIR / "day16_novel_predictions"
DAY16_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("DEVICE:", DEVICE)

2 — Load گراف و فایل‌ها

In [ ]:
nodes = pd.read_csv(GRAPH_DIR / "graph_nodes.csv")
interaction_edges = pd.read_csv(GRAPH_DIR / "interaction_edges_labeled.csv")
ppi_edges = pd.read_csv(GRAPH_DIR / "ppi_edges.csv")
loc_edges = pd.read_csv(GRAPH_DIR / "colocalization_edges.csv")
node_features = np.load(GRAPH_DIR / "node_features_esm650.npy")

print("nodes:", nodes.shape)
print("interaction_edges:", interaction_edges.shape)
print("ppi_edges:", ppi_edges.shape)
print("loc_edges:", loc_edges.shape)
print("node_features:", node_features.shape)

display(nodes.head())
display(interaction_edges.head())

3 — ساخت HeteroData برای inference

In [ ]:
def edge_index_from_df(df):
    return torch.tensor(
        df[["src", "dst"]].values.T,
        dtype=torch.long
    )

hetero_data = HeteroData()
hetero_data["protein"].x = torch.tensor(node_features, dtype=torch.float32)

# enzyme-substrate known graph context
hetero_data["protein", "enzyme_substrate", "protein"].edge_index = edge_index_from_df(
    interaction_edges
)

# ppi undirected
ppi_edge_index = edge_index_from_df(ppi_edges)
ppi_edge_index = torch.cat(
    [ppi_edge_index, ppi_edge_index[[1, 0], :]],
    dim=1
)
hetero_data["protein", "ppi", "protein"].edge_index = ppi_edge_index

# co-localization undirected
loc_edge_index = edge_index_from_df(loc_edges)
loc_edge_index = torch.cat(
    [loc_edge_index, loc_edge_index[[1, 0], :]],
    dim=1
)
hetero_data["protein", "co_localized", "protein"].edge_index = loc_edge_index

hetero_data = hetero_data.to(DEVICE)

print(hetero_data)

Cell 4 — Load metadata از checkpoint

In [ ]:
ckpt0 = torch.load(
    MODEL_DIR / "hgt_fold0_best.pt",
    map_location="cpu"
)

HGT_METADATA = ckpt0["metadata"]

print("metadata:", HGT_METADATA)
print("in_dim:", ckpt0["in_dim"])
print("hidden_dim:", ckpt0["hidden_dim"])
print("emb_dim:", ckpt0["emb_dim"])
print("heads:", ckpt0["heads"])
print("dropout:", ckpt0["dropout"])

5 — تعریف مدل HGT

In [ ]:
class HGTEncoder(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        out_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.lin_in = nn.Linear(in_dim, hidden_dim)

        self.hgt1 = HGTConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            metadata=HGT_METADATA,
            heads=heads,
        )

        self.hgt2 = HGTConv(
            in_channels=hidden_dim,
            out_channels=out_dim,
            metadata=HGT_METADATA,
            heads=heads,
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(out_dim)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict):
        x = x_dict["protein"]

        x = self.lin_in(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.hgt1(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm1(x)
        x = F.gelu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = {"protein": x}

        x_dict = self.hgt2(x_dict, edge_index_dict)
        x = x_dict["protein"]
        x = self.norm2(x)
        x = F.gelu(x)

        return {"protein": x}


class HeteroLinkPredictor(nn.Module):
    def __init__(self, emb_dim=128, hidden_dim=128, dropout=0.35):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, z_dict, edge_label_index):
        z = z_dict["protein"]

        src = edge_label_index[0]
        dst = edge_label_index[1]

        z_src = z[src]
        z_dst = z[dst]

        h = torch.cat(
            [
                z_src,
                z_dst,
                torch.abs(z_src - z_dst),
                z_src * z_dst,
            ],
            dim=1,
        )

        return self.mlp(h).squeeze(-1)


class HGTLinkModel(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=128,
        emb_dim=128,
        heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.encoder = HGTEncoder(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=emb_dim,
            heads=heads,
            dropout=dropout,
        )

        self.predictor = HeteroLinkPredictor(
            emb_dim=emb_dim,
            hidden_dim=128,
            dropout=dropout,
        )

    def forward(self, data, edge_label_index):
        edge_index_dict = {
            k: v
            for k, v in data.edge_index_dict.items()
            if k in HGT_METADATA[1]
        }

        z_dict = self.encoder(
            data.x_dict,
            edge_index_dict,
        )

        logits = self.predictor(
            z_dict,
            edge_label_index,
        )

        return logits

6 — استخراج همه E3ها، DUBها و Substrateها

In [ ]:
nodes["gene"] = nodes["gene"].astype(str)
nodes["uniprot_ac"] = nodes["uniprot_ac"].astype(str)

e3_nodes = nodes[nodes["is_e3"] == 1].copy()
dub_nodes = nodes[nodes["is_dub"] == 1].copy()
sub_nodes = nodes[nodes["is_substrate"] == 1].copy()

print("E3 nodes:", e3_nodes.shape)
print("DUB nodes:", dub_nodes.shape)
print("Substrate nodes:", sub_nodes.shape)

display(e3_nodes.head())
display(dub_nodes.head())
display(sub_nodes.head())

7 — ساخت همه زوج‌های ممکن

In [ ]:
def build_candidate_pairs(enzyme_df, substrate_df, enzyme_class):
    rows = []

    for _, e in enzyme_df.iterrows():
        e_node = int(e["node_id"])
        e_ac = str(e["uniprot_ac"])
        e_gene = str(e["gene"])

        for _, s in substrate_df.iterrows():
            s_node = int(s["node_id"])
            s_ac = str(s["uniprot_ac"])
            s_gene = str(s["gene"])

            if e_node == s_node:
                continue

            pair_id = f"{enzyme_class}|{e_ac}|{s_ac}"

            rows.append({
                "pair_id": pair_id,
                "enzyme_class": enzyme_class,
                "enz_node": e_node,
                "sub_node": s_node,
                "enz_ac": e_ac,
                "sub_ac": s_ac,
                "enz_gene": e_gene,
                "sub_gene": s_gene,
            })

    return pd.DataFrame(rows)


e3_candidates = build_candidate_pairs(
    e3_nodes,
    sub_nodes,
    "E3"
)

dub_candidates = build_candidate_pairs(
    dub_nodes,
    sub_nodes,
    "DUB"
)

all_candidates = pd.concat(
    [e3_candidates, dub_candidates],
    ignore_index=True
)

print("E3 candidates:", e3_candidates.shape)
print("DUB candidates:", dub_candidates.shape)
print("All candidates:", all_candidates.shape)

display(all_candidates.head())

8 — حذف Known Pairs

In [ ]:
known_pair_ids = set(interaction_edges["pair_id"].astype(str))

all_candidates["is_known_pair"] = all_candidates["pair_id"].isin(known_pair_ids)

print("known in candidate space:", all_candidates["is_known_pair"].sum())
print("candidate before remove:", all_candidates.shape)

novel_candidates = (
    all_candidates[
        ~all_candidates["is_known_pair"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("novel candidates:", novel_candidates.shape)

novel_candidates.to_csv(
    DAY16_DIR / "novel_candidate_space_unscored.csv",
    index=False
)

display(novel_candidates.head())

9 — تابع scoring با یک HGT checkpoint

In [ ]:
def load_hgt_model(fold):
    ckpt = torch.load(
        MODEL_DIR / f"hgt_fold{fold}_best.pt",
        map_location="cpu"
    )

    model = HGTLinkModel(
        in_dim=ckpt["in_dim"],
        hidden_dim=ckpt["hidden_dim"],
        emb_dim=ckpt["emb_dim"],
        heads=ckpt["heads"],
        dropout=ckpt["dropout"],
    ).to(DEVICE)

    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    return model


def score_candidates_with_model(
    model,
    candidates_df,
    batch_size=8192,
):
    probs = []

    model.eval()

    for start in range(0, len(candidates_df), batch_size):
        end = min(start + batch_size, len(candidates_df))

        batch = candidates_df.iloc[start:end]

        edge_index = torch.tensor(
            batch[["enz_node", "sub_node"]].values.T,
            dtype=torch.long,
            device=DEVICE
        )

        with torch.no_grad():
            logits = model(
                hetero_data,
                edge_index
            )
            prob = torch.sigmoid(logits).detach().cpu().numpy()

        probs.append(prob)

        if start % (batch_size * 10) == 0:
            print(f"{start:,} / {len(candidates_df):,}")

    return np.concatenate(probs)

10 — scoring با ۵با  fold و میانگین‌گیری

In [ ]:
scored = novel_candidates.copy()

for fold in range(5):
    print("=" * 100)
    print("Scoring fold:", fold)

    model = load_hgt_model(fold)

    scored[f"prob_hgt_fold{fold}"] = score_candidates_with_model(
        model,
        scored,
        batch_size=8192,
    )

    del model
    torch.mps.empty_cache() if DEVICE.type == "mps" else None

scored["prob_hgt_mean"] = scored[
    [f"prob_hgt_fold{i}" for i in range(5)]
].mean(axis=1)

scored["prob_hgt_std"] = scored[
    [f"prob_hgt_fold{i}" for i in range(5)]
].std(axis=1)

scored = scored.sort_values(
    "prob_hgt_mean",
    ascending=False
).reset_index(drop=True)

display(scored.head(20))

scored.to_csv(
    DAY16_DIR / "novel_candidate_space_hgt_scored.csv",
    index=False
)

11 — استخراج Top Novel Predictions

In [ ]:
top50 = scored.head(50).copy()
top100 = scored.head(100).copy()
top500 = scored.head(500).copy()

top50.to_csv(DAY16_DIR / "top50_novel_predictions_hgt.csv", index=False)
top100.to_csv(DAY16_DIR / "top100_novel_predictions_hgt.csv", index=False)
top500.to_csv(DAY16_DIR / "top500_novel_predictions_hgt.csv", index=False)

display(
    top50[
        [
            "enzyme_class",
            "enz_gene",
            "sub_gene",
            "prob_hgt_mean",
            "prob_hgt_std",
            "pair_id"
        ]
    ]
)

12 — QC نهایی

In [ ]:
qc = {
    "n_e3_nodes": len(e3_nodes),
    "n_dub_nodes": len(dub_nodes),
    "n_substrate_nodes": len(sub_nodes),
    "n_all_candidates": len(all_candidates),
    "n_known_removed": int(all_candidates["is_known_pair"].sum()),
    "n_novel_candidates": len(novel_candidates),
    "top50_min_prob": top50["prob_hgt_mean"].min(),
    "top100_min_prob": top100["prob_hgt_mean"].min(),
    "top500_min_prob": top500["prob_hgt_mean"].min(),
}

qc_df = pd.DataFrame([qc])
display(qc_df)

qc_df.to_csv(
    DAY16_DIR / "day16_novel_prediction_qc.csv",
    index=False
)

qc = {
    "n_e3_nodes": len(e3_nodes),
    "n_dub_nodes": len(dub_nodes),
    "n_substrate_nodes": len(sub_nodes),
    "n_all_candidates": len(all_candidates),
    "n_known_removed": int(all_candidates["is_known_pair"].sum()),
    "n_novel_candidates": len(novel_candidates),
    "top50_min_prob": top50["prob_hgt_mean"].min(),
    "top100_min_prob": top100["prob_hgt_mean"].min(),
    "top500_min_prob": top500["prob_hgt_mean"].min(),
}

qc_df = pd.DataFrame([qc])
display(qc_df)

qc_df.to_csv(
    DAY16_DIR / "day16_novel_prediction_qc.csv",
    index=False
)